In [4]:
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

sklepy = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
kategorie = ['elektronika', 'odzież', 'żywność', 'książki']

def generate_transaction():
    if random.random() < 0.05:
        return { #podejrzana
            'tx_id': f'TX{random.randint(1000,9999)}',
            'user_id': f'u{random.randint(1,20):02d}',
            'amount': round(random.uniform(3000.0, 5000.0), 2),
            'store': random.choice(sklepy),
            'category': 'elektronika',
            'hour': random.randint(0,5),
            'timestamp': datetime.now().isoformat(),
            'is_suspicious': True
        }
    else: #normalna
        return {
            'tx_id': f'TX{random.randint(1000,9999)}',
            'user_id': f'u{random.randint(1,20):02d}',
            'amount': round(random.uniform(5.0, 5000.0), 2),
            'store': random.choice(sklepy),
            'category': random.choice(kategorie),
            'hour': random.randint(0,23),
            'timestamp': datetime.now().isoformat(),
            'is_suspicious': False
        }
for i in range(1000):
    tx = generate_transaction()
    producer.send('transactions', value=tx)
    label = "PODEJRZANA" if tx['is_suspicious'] else ""
    print(f"[{i+1}] {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']} | {tx['category']} | h:{tx['hour']} {label}")
    time.sleep(0.5)

producer.flush()
producer.close()

Overwriting producer.py


In [8]:
%%file consumer_filter.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)
for message in consumer:
    tx = message.value
    
    if tx['amount'] > 1000:
        print(f"ALERT: {tx['tx_id']} | {tx['amount']} PLN | {tx['store']}")

Overwriting consumer_filter.py


In [9]:
%%file consumer_enrich.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

for message in consumer:
    tx = message.value

    if tx['amount'] > 3000:
        risk = "HIGH"
    elif tx['amount'] > 1000:
        risk = "MEDIUM"
    else:
        risk = "LOW"
    
    print(f"{tx['tx_id']} | {tx['amount']} PLN | {tx['store']} | risk: {risk}")

Writing consumer_enrich.py


In [10]:
%%file consumer_count.py
from kafka import KafkaConsumer
from collections import Counter, defaultdict
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='count-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

store_counts = Counter()
total_amount = defaultdict(float)
msg_count = 0

for message in consumer:
    tx = message.value
    
    store = tx['store']
    amount = tx['amount']

    store_counts[store] += 1
    total_amount[store] += amount
    
    msg_count += 1
    
    if msg_count % 10 == 0:
        print("\n--- PODSUMOWANIE ---")
        for s in store_counts:
            print(f"{s}: count={store_counts[s]}, total={total_amount[s]:.2f} PLN")

Writing consumer_count.py


In [11]:
def score_transaction(tx):
    score = 0
    rules = []
    
    if tx['amount'] > 3000:
        score += 3
        rules.append('R1')
    
    if tx['category'] == 'elektronika' and tx['amount'] > 1500:
        score += 2
        rules.append('R2')
    
    if tx.get('hour', 12) < 6:
        score += 2
        rules.append('R3')
    
    return score, rules

In [13]:
# Test
test_tx = {'tx_id': 'TX999', 'amount': 4500.0, 'category': 'elektronika',
           'timestamp': '2026-04-01T03:15:00'}
print(score_transaction(test_tx))  # powinno dać score >= 5

(5, ['R1', 'R2'])


In [15]:
%%file scoring_consumer.py
from kafka import KafkaConsumer, KafkaProducer
import json

consumer = KafkaConsumer('transactions', bootstrap_servers='broker:9092',
    auto_offset_reset='earliest', group_id='scoring-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8')))

alert_producer = KafkaProducer(bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8'))

def score_transaction(tx):
    score = 0
    rules = []
    
    if tx['amount'] > 3000:
        score += 3
        rules.append('R1')
    
    if tx['category'] == 'elektronika' and tx['amount'] > 1500:
        score += 2
        rules.append('R2')
    
    if tx.get('hour', 12) < 6:
        score += 2
        rules.append('R3')
    
    return score, rules
    
for message in consumer:
    tx = message.value
    
    score, rules = score_transaction(tx)
    
    if score >= 3:
        alert_producer.send('alerts', value={
            'tx': tx,
            'score': score,
            'rules': rules
        })
        
        print(f"ALERT: {tx['tx_id']} | score={score} | rules={rules}")

Overwriting scoring_consumer.py
